In [4]:
import os
import sys
import openai
from dotenv import load_dotenv
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_openai import ChatOpenAI
from pathlib import Path

In [5]:
from google.oauth2 import service_account
from googleapiclient.discovery import build

# Auth
SERVICE_ACCOUNT_FILE = '/Users/daianeklein/Documents/DS/job-applications-tool/h.json'
SCOPES = ["https://www.googleapis.com/auth/documents.readonly"]
creds = service_account.Credentials.from_service_account_file(SERVICE_ACCOUNT_FILE, scopes=SCOPES)

# Docs API Service
service = build('docs', 'v1', credentials=creds)

DOCUMENT_ID = '1pXc4nsuFd5RfQFWimmKaLMxucWiV7WLZDCtP18wbCTE'

def fetch_cv_text():
    """Fetches the CV document content and extracts text."""
    doc = service.documents().get(documentId=DOCUMENT_ID).execute()

    def extract_text(document):
        text = []
        for element in document.get("body", {}).get("content", []):
            if "paragraph" in element:
                for paragraph_element in element["paragraph"]["elements"]:
                    if "textRun" in paragraph_element:
                        text.append(paragraph_element["textRun"]["content"])
        return "".join(text)

    return extract_text(doc)


if __name__ == '__main__':
    document_text = fetch_cv_text()
    print("\nDocument Content:\n", document_text)



Document Content:
 Daiane Klein
Senior Data Analyst | Analytics Engineer

São Paulo, Brazil    |   +55 11 962192070    |    Linkedin    |     Github
PROFILE SUMMARY
Data Analyst with 7+ years of experience in Data Analysis, including business and customer insights, in different industries. Proficient in Python, SQL, and Dashboard development. Strong understanding of Machine Learning, statistics, and Large Language Models (LLMs) as well as business impact and results.

 	SKILLS
	Professional Skills: 	Data Analysis | Data Science | Data Visualization | ETL | Data  Engineering
	Stacks & Tools: 	Python | SQL | Power BI | Cloud | Docker | Snowflake | dbt | Git
Languages: 		Portuguese (Native) | English (C1 Advanced)

 	WORK EXPERIENCE
	AI & Data Strategy Consultant, Stealth AI Startup  (Contract)					Jan 2025 - Present
Developed an LLM-powered pipeline to analyze unstructured data and generate business insights.
Collaborated directly with customers to refine solutions, understand customer 

In [6]:
load_dotenv()
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')

if not OPENAI_API_KEY:
    raise ValueError('OPENAI API KEY NOT FOUND')

# Initialize OpenAI Chat Model
llm = ChatOpenAI(model_name='gpt-4o', openai_api_key=OPENAI_API_KEY)

In [55]:
doc = service.documents().get(documentId=DOCUMENT_ID).execute()
doc

{'title': 'daiane-klein-resume-template',
 'body': {'content': [{'endIndex': 1,
    'sectionBreak': {'sectionStyle': {'columnSeparatorStyle': 'NONE',
      'contentDirection': 'LEFT_TO_RIGHT',
      'sectionType': 'CONTINUOUS'}}},
   {'startIndex': 1,
    'endIndex': 14,
    'paragraph': {'elements': [{'startIndex': 1,
       'endIndex': 14,
       'textRun': {'content': 'Daiane Klein\n',
        'textStyle': {'bold': True,
         'fontSize': {'magnitude': 22, 'unit': 'PT'},
         'weightedFontFamily': {'fontFamily': 'Lato', 'weight': 400}}}}],
     'paragraphStyle': {'namedStyleType': 'NORMAL_TEXT',
      'alignment': 'CENTER',
      'lineSpacing': 115,
      'direction': 'LEFT_TO_RIGHT',
      'spacingMode': 'COLLAPSE_LISTS',
      'spaceAbove': {'unit': 'PT'},
      'spaceBelow': {'unit': 'PT'},
      'borderBetween': {'color': {},
       'width': {'unit': 'PT'},
       'padding': {'unit': 'PT'},
       'dashStyle': 'SOLID'},
      'borderTop': {'color': {},
       'width': {'u

In [18]:
def fetch_job_title(doc:str) -> str:
    """Fetches the job title from the CV document in Google Docs."""    
    content = doc.get("body", {}).get("content", [])
    print('\n\n------\n\n')
    print(content)
    print('\n\n------\n\n')
    
    job_title = None
    paragraph_count = 0  # Track which paragraph we're processing

    for element in content:
        if "paragraph" in element:
            paragraph_count += 1  # Increment for each paragraph
            
            # The second paragraph should be the job title
            if paragraph_count == 2:
                for paragraph_element in element["paragraph"]["elements"]:
                    if "textRun" in paragraph_element:
                        job_title = paragraph_element["textRun"]["content"].strip()
                break  # Stop after finding the second paragraph

    return job_title

In [39]:
content = doc.get("body", {}).get("content", [])
content2 = content[2]['paragraph']['elements'][0]['textRun']['content']

content2

'chata'

In [43]:
def fetch_job_title(doc: str) -> str:
    content = doc.get("body", {}).get("content", [])

    job_title = content[2]['paragraph']['elements'][0]['textRun']['content']

    return job_title


In [44]:
fetch_job_title(doc)

'Senior Data Analyst | Analytics Engineer\n'

In [ ]:
def fetch_profile_summary(doc: str) -> str:
    content = doc.get("body", {}).get("content", [])

    job_title = content[2]['paragraph']['elements'][0]['textRun']['content']

    return job_title


In [67]:
content = doc.get("body", {}).get("content", [])

content[9]['paragraph']['elements'][2]['textRun']['content'].strip()

'Data Analysis | Data Science | Data Visualization | ETL | Data  Engineering'